In [1]:
import random
from rdkit import Chem
from molpher.core import MolpherMol, MolpherAtom
from molpher.core.morphing.operators import MorphingOperator
from rdkit.Chem.EnumerateStereoisomers import EnumerateStereoisomers, StereoEnumerationOptions
from rdkit.Chem import rdChemReactions
from rdkit.Chem import rdmolops
from rdkit.Chem import Descriptors  
from molpher.core import ExplorationTree as ETree

class AldehydeReduction(MorphingOperator):
    def __init__(self):
        super(AldehydeReduction, self).__init__()
        self._name = "Aldehyde Reduction (Phase I - Safe)"
        self._matches = []
        self.ALDEHYDE_PATTERN = Chem.MolFromSmarts("[CX3H1;!$(C(=O)[O,N,S,F,Cl,Br,I])]=[OX1]")

    def setOriginal(self, mol):
        super(AldehydeReduction, self).setOriginal(mol)
        self._matches = []

        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return

        matches = rdkit_mol.GetSubstructMatches(self.ALDEHYDE_PATTERN)
        for match in matches:
            self._matches.append((match[0], match[1]))

    def morph(self):
        if not self.original: return None
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return None

        if not self._matches:
            return MolpherMol(other=rdkit_mol)

        edit_mol = Chem.RWMol(rdkit_mol)
        c_idx, o_idx = random.choice(self._matches)

        try:
            bond = edit_mol.GetBondBetweenAtoms(c_idx, o_idx)
            if bond is None: return MolpherMol(other=rdkit_mol)

            bond.SetBondType(Chem.BondType.SINGLE)

            for idx in [c_idx, o_idx]:
                atom = edit_mol.GetAtomWithIdx(idx)
                atom.SetFormalCharge(0)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                for prop in list(atom.GetPropNames()):
                    atom.ClearProp(prop)
                atom.UpdatePropertyCache(strict=False)

            new_mol = edit_mol.GetMol()
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name

aldehyde_op = AldehydeReduction()

class KetoneReduction(MorphingOperator):
    def __init__(self):
        super(KetoneReduction, self).__init__()
        self._name = "Ketone Reduction (Phase I - Stereospecific Enumerate)"
        self._matches = []

    def setOriginal(self, mol):
        super(KetoneReduction, self).setOriginal(mol)
        self._matches = []

        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return
        
        ketone_pattern = Chem.MolFromSmarts("[CX3;$(C(=O)(-[#6])-[#6])]=O")
        matches = rdkit_mol.GetSubstructMatches(ketone_pattern)
        for match in matches:
            self._matches.append((match[0], match[1]))

    def morph(self):
        if not self.original: return None
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return None

        if not self._matches:
            return MolpherMol(other=rdkit_mol)

        edit_mol = Chem.RWMol(rdkit_mol)
        carbonyl_idx, oxygen_idx = random.choice(self._matches)

        try:
            bond = edit_mol.GetBondBetweenAtoms(carbonyl_idx, oxygen_idx)
            if bond is None: return MolpherMol(other=rdkit_mol)

            bond.SetBondType(Chem.BondType.SINGLE)

            for idx in [carbonyl_idx, oxygen_idx]:
                atom = edit_mol.GetAtomWithIdx(idx)
                atom.SetFormalCharge(0)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                atom.SetChiralTag(Chem.ChiralType.CHI_UNSPECIFIED)
                if atom.HasProp('_CIPCode'): atom.ClearProp('_CIPCode')
                for prop in list(atom.GetPropNames()): atom.ClearProp(prop)
                atom.UpdatePropertyCache(strict=False)

            new_mol = edit_mol.GetMol()
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)

            options = StereoEnumerationOptions(onlyUnassigned=True)
            isomers = list(EnumerateStereoisomers(new_mol, options=options))

            if isomers:
                chosen_iso = random.choice(isomers)
                Chem.SanitizeMol(chosen_iso)
                Chem.AssignStereochemistry(chosen_iso, cleanIt=True, force=True)
                return MolpherMol(other=chosen_iso)

            return MolpherMol(other=new_mol)
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name


ketone_op = KetoneReduction()

start_smiles = "CC(=O)CC(=O)O.O=Cc1ccccc1" 
root_mol = MolpherMol(start_smiles)

print(f"GENERATION 0 (Root Node):\n  SMILES: {root_mol.getSMILES()}\n")

# --- ΓΕΝΙΑ 1: Εφαρμογή Aldehyde Reduction στον Root μόριο ---
aldehyde_op.setOriginal(root_mol)
gen1_mol = aldehyde_op.morph()

print(f"GENERATION 1 (Aldehyde Reduction -> Primary Alcohol):")
print(f"  SOURCE: {root_mol.getSMILES()}")
print(f"  TARGET: {gen1_mol.getSMILES() if gen1_mol else 'Failed'}")
# Έλεγχος αν προστατεύτηκε το οξύ CC(=O)O
if "C(O)O" in gen1_mol.getSMILES() or "C(=O)O" in gen1_mol.getSMILES():
    print("  PROTECTION CHECK: Passed (Carboxylic acid was safely ignored!)")
print("\n")

# --- ΓΕΝΙΑ 2: Παίρνουμε το προϊόν της Γενιάς 1 και του κάνουμε Ketone Reduction
ketone_op.setOriginal(gen1_mol)
gen2_mol = ketone_op.morph()

print(f"GENERATION 2 (Ketone Reduction -> Chiral Secondary Alcohol):")
print(f"  SOURCE: {gen1_mol.getSMILES()}")
print(f"  TARGET: {gen2_mol.getSMILES() if gen2_mol else 'Failed'}")

GENERATION 0 (Root Node):
  SMILES: CC(=O)CC(=O)O.O=CC1=CC=CC=C1

GENERATION 1 (Aldehyde Reduction -> Primary Alcohol):
  SOURCE: CC(=O)CC(=O)O.O=CC1=CC=CC=C1
  TARGET: CC(=O)CC(=O)O.OCC1=CC=CC=C1
  PROTECTION CHECK: Passed (Carboxylic acid was safely ignored!)


GENERATION 2 (Ketone Reduction -> Chiral Secondary Alcohol):
  SOURCE: CC(=O)CC(=O)O.OCC1=CC=CC=C1
  TARGET: CC(O)CC(=O)O.OCC1=CC=CC=C1
